In [1]:
#pip install torch torchvision torchaudio

In [2]:
import torch
from torch import nn


In [3]:
# setup for my m1pro laptop
if torch.backends.mps.is_available():
    device = torch.device("mps") # Apple Silicon GPU
elif torch.cuda.is_available():
    device = torch.device("cuda") # NVIDIA GPU - if available
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [4]:
class BrainTumorCNN(nn.Module):
    """
    Improved CNN for brain tumor classification.

    Key differences vs old SimpleCNN:
    - Multi-scale first block: 3x3 and 5x5 conv branches in parallel.
    - Deeper conv blocks so receptive field covers larger brain regions (better for asymmetry).
    - Texture-focused final block with more channels.
    - Global Average Pooling instead of huge Flatten+Linear (fewer params).
    - Still lightweight enough to train on an M1 Pro chip.
    """
    def __init__(self, in_channels: int = 3, num_classes: int = 1):
        # in_channels: number of input channels (3 for RGB)
        # num_classes: number of output classes (1 for binary classification)
        super().__init__()

        # -------- Block 1: Multi-scale feature extraction --------
        # 3x3 conv ( edges, boundaries)
        # 5x5 conv ( larger structures, coarse tumor shape)
        # 2 branches in parallel to capture different scales of features
        self.branch3x3 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        # Branch B: 5x5 conv (larger structures, coarse tumor shape)
        self.branch5x5 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=5, padding=2),
            nn.ReLU()
        )

        # After concatenation, channels = 16 + 16 = 32
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 64x64 -> 32x32

        # -------- Block 2: Deeper receptive field --------
        # Two stacked 3x3 convs increase effective receptive field, which helps capture asymmetry or larger patterns of tumors
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # 32x32 -> 16x16
        )

        # -------- Block 3: Texture-focused block --------
        # More channels, still 3x3, no more spatial downsampling
        # Designed to learn tumor texture (heterogeneity, contrast) which is the overall appearance
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # -------- Global Average Pooling + Classifier --------
        # Global Average Pooling: each channel -> 1 scalar (how strong that pattern is anywhere)
        # benefits: fewer parameters, less overfitting, spatial invariance
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))  # (N, C, H, W) -> (N, C, 1, 1)

        self.classifier = nn.Sequential(
            nn.Flatten(),               # (N, 128, 1, 1) -> (N, 128)
            nn.Dropout(p=0.5),          # Regularization (important for small-ish dataset)
            nn.Linear(128, 1)           # Single logit for BCEWithLogitsLoss
        )

    def forward(self, x):
        # x: (N, 3, 64, 64)

        # Multi-scale block
        x3 = self.branch3x3(x)        
        x5 = self.branch5x5(x)        
        x = torch.cat([x3, x5], dim=1)

        x = self.pool1(x)             

        # run both branches and concatenate
        # Deeper block
        x = self.block2(x)            

        # Texture block
        x = self.block3(x)            

        # Global average pooling
        x = self.global_pool(x)       

        # Classifier
        x = self.classifier(x) # (N, 1)

        return x

In [5]:
model = BrainTumorCNN(in_channels=3, num_classes=1).to(device)
print(model)

BrainTumorCNN(
  (branch3x3): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
  )
  (branch5x5): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
  )
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
  )
  (global_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, i

In [6]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import time

In [7]:
data_path = "../dataset_with_label"

In [8]:
# -------- Transforms --------
train_transform = transforms.Compose([
    transforms.Resize((64, 64)), # resize to 64x64 so model input size matches
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(), # convert PIL image to Tensor
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3) # dataset normalization
])

test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# ---- Load ImageFolder datasets ----
train_data = datasets.ImageFolder(root=f"{data_path}/train", transform=train_transform)
test_data  = datasets.ImageFolder(root=f"{data_path}/test",  transform=test_transform)

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))

train_dataloader = DataLoader(train_data, batch_size=16, shuffle=True)
test_dataloader  = DataLoader(test_data,  batch_size=16, shuffle=False)

print("Dataloaders ready.")

Train samples: 3009
Test samples: 753
Dataloaders ready.


In [9]:
'''class BrainTumorCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=1):
        super().__init__()

        # multiple filter sizes in the first block(kernal size 3 and 5)
        # those 2 kernal will slide over the image in parallel to extract features at different scales
        self.branch3x3 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.branch5x5 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=5, padding=2),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2, 2)

        # second block(kernal) with increased detection field
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # a very large number of channels to focus on texture which is the overall of the image
        # eg. find smoothness, roughness, regular patterns
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # combined with global average pooling and dropout to reduce overfitting
        # so each channel is summarized into one value indicating the presence of that feature anywhere in the image
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x3 = self.branch3x3(x)
        x5 = self.branch5x5(x)
        x = torch.cat([x3, x5], dim=1)

        x = self.pool1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x'''

'class BrainTumorCNN(nn.Module):\n    def __init__(self, in_channels=3, num_classes=1):\n        super().__init__()\n\n        # multiple filter sizes in the first block(kernal size 3 and 5)\n        # those 2 kernal will slide over the image in parallel to extract features at different scales\n        self.branch3x3 = nn.Sequential(\n            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),\n            nn.ReLU()\n        )\n        self.branch5x5 = nn.Sequential(\n            nn.Conv2d(in_channels, 16, kernel_size=5, padding=2),\n            nn.ReLU()\n        )\n        self.pool1 = nn.MaxPool2d(2, 2)\n\n        # second block(kernal) with increased detection field\n        self.block2 = nn.Sequential(\n            nn.Conv2d(32, 64, kernel_size=3, padding=1),\n            nn.ReLU(),\n            nn.Conv2d(64, 64, kernel_size=3, padding=1),\n            nn.ReLU(),\n            nn.MaxPool2d(2, 2)\n        )\n\n        # a very large number of channels to focus on texture which

In [ ]:
# ----- Single epoch training -----
def train_step(model, dataloader, loss_fn, optimizer, device):
    ''' Performs a single epoch training step 
    arguments:
    model: the neural network model
    dataloader: DataLoader for training data
    loss_fn: loss function
    optimizer: optimizer for updating model parameters
    device: computation device (cpu, cuda, mps)
    returns:
    average loss, average accuracy, time taken for the epoch'''
    model.train() # this is for training mode
    total_loss, total_acc = 0, 0
    start = time.time()

    for X, y in dataloader:
        X, y = X.to(device), y.to(device).float() # mps and cuda need float labels for BCEWithLogitsLoss

        logits = model(X).squeeze(1)
        loss = loss_fn(logits, y)

        # update model parameters by backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # logits to predictions, when sigmoid>0.5 -> class 1 else class 0
        preds = (torch.sigmoid(logits) > 0.5).int()
        total_acc += (preds == y.int()).sum().item() / len(y)

    return total_loss / len(dataloader), total_acc / len(dataloader), time.time() - start


# ----- Single epoch testing -----
def test_step(model, dataloader, loss_fn, device):
    ''' Performs a single epoch testing/validation step
    arguments:
    model: the neural network model
    dataloader: DataLoader for testing/validation data
    loss_fn: loss function
    device: computation device (cpu, cuda, mps)
    returns:
    average loss, average accuracy, time taken for the epoch'''
    model.eval()
    total_loss, total_acc = 0, 0
    start = time.time()

    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device).float()
            logits = model(X).squeeze(1)
            loss = loss_fn(logits, y)

            total_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).int()
            total_acc += (preds == y.int()).sum().item() / len(y)

    return total_loss / len(dataloader), total_acc / len(dataloader), time.time() - start


# ----- Train Loop -----
def train(model, train_dataloader, test_dataloader, optimizer, loss_fn, device, epochs=10):
    ''' Full training loop over multiple epochs, combines train_step and test_step 
    Loss(BCEWithLogitsLoss) is what the optimizer minimizes '''
    for epoch in range(epochs):
        train_loss, train_acc, t_time = train_step(model, train_dataloader, loss_fn, optimizer, device)
        test_loss, test_acc, v_time  = test_step(model, test_dataloader, loss_fn, device)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train: loss={train_loss:.4f}, acc={train_acc:.4f}, time={t_time:.2f}s | "
            f"Test: loss={test_loss:.4f}, acc={test_acc:.4f}, time={v_time:.2f}s"
        )

In [11]:
model = BrainTumorCNN().to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=10
)

Epoch 1/10 | Train: loss=0.5850, acc=0.6839, time=6.07s | Test: loss=0.4592, acc=0.8021, time=0.70s
Epoch 2/10 | Train: loss=0.4288, acc=0.8142, time=5.38s | Test: loss=0.6227, acc=0.6510, time=0.69s
Epoch 3/10 | Train: loss=0.4199, acc=0.8241, time=4.89s | Test: loss=0.3390, acc=0.8620, time=0.74s
Epoch 4/10 | Train: loss=0.3272, acc=0.8638, time=4.49s | Test: loss=0.2830, acc=0.8854, time=0.53s
Epoch 5/10 | Train: loss=0.2611, acc=0.8942, time=4.26s | Test: loss=0.2578, acc=0.8945, time=0.54s
Epoch 6/10 | Train: loss=0.2597, acc=0.9058, time=4.21s | Test: loss=0.2323, acc=0.9036, time=0.54s
Epoch 7/10 | Train: loss=0.2166, acc=0.9233, time=4.22s | Test: loss=0.2337, acc=0.9258, time=0.53s
Epoch 8/10 | Train: loss=0.2083, acc=0.9226, time=4.30s | Test: loss=0.1985, acc=0.9141, time=0.72s
Epoch 9/10 | Train: loss=0.1985, acc=0.9312, time=4.45s | Test: loss=0.1708, acc=0.9336, time=0.58s
Epoch 10/10 | Train: loss=0.1824, acc=0.9345, time=4.42s | Test: loss=0.1658, acc=0.9414, time=0.55s